In [ ]:
import pandas as pd 
df = pd.read_csv('./customer_shopping_behavior.csv')

In [ ]:
df.head()

In [ ]:
# structure
df.info()
# Review Rating - 3863 non-null   float64

In [ ]:
# summary statistics
df.describe() # numerical columns 

In [ ]:
df.describe(include = 'all') # all columns

In [ ]:
# missing values
df.isnull().sum()
# Review Rating  37

# handle missing values
## fill with median over mean -
  * mean can get affected by the outliers and that can skew the distribution of the data
  * median is robust to outliers
  * median of each category -> to avoid bias into the dataset

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum()

In [ ]:
# modify - snake casing - easy referencing of code in sql / python
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.rename(columns = {'purchase_amount_(usd)' : 'purchase_amount_usd'})

In [ ]:
df.columns

## feature enginnering - create new columns


In [ ]:
# create column - age_group
label = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q = 4, labels = label)

In [ ]:
df[['age', 'age_group']].head(10)

In [ ]:
df['frequency_of_purchases'].unique()

In [ ]:
# create column - purchase_frequency_days - numbers
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Annually': 365,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Monthly': 30,
    'Every 3 Months': 90
}
# now handling inconsistencies is not req.

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

In [ ]:
# handling redundency
df[['discount_applied', 'promo_code_used']]

In [ ]:
bool((df['discount_applied'] == df['promo_code_used']).all())

In [ ]:
df = df.drop('promo_code_used', axis = 1)

In [ ]:
df.columns

## Connecting Python script to MySQL

In [ ]:
!pip install pymysql sqlalchemy

In [ ]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import pandas as pd

username = "root"
password = quote_plus("mysql@alka")   # encode special chars
host = "localhost"
port = "3306"
database = "customer_behavior"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

# Write dataframe
df.to_sql("customer", engine, if_exists="replace", index=False)

# Read sample
pd.read_sql("SELECT * FROM customer LIMIT 10;", engine)